# سبق 12 - ایجنٹ اسکریچ پیڈ کے ساتھ چیٹ ہسٹری کی کمی

یہ نوٹ بک دکھاتی ہے کہ مائیکروسافٹ ایجنٹ فریم ورک کا استعمال کرتے ہوئے لمبی گفتگو میں سیاق و سباق کو کیسے سنبھالا جا سکتا ہے۔ جیسے جیسے گفتگو بڑھتی ہے، ٹوکن کی تعداد بڑھ جاتی ہے — بالآخر ماڈل کی سیاق و سباق کی ونڈو سے تجاوز کر جاتی ہے۔ ہم اس مسئلے کو **سیاق و سباق کو مختصر کرنے کے پیٹرن** اور **ایجنٹ اسکریچ پیڈ** کے ذریعے مستقل یادداشت کے ساتھ حل کرتے ہیں۔

## آپ کیا سیکھیں گے:
1. **سیاق و سباق کے انتظام کی اہمیت**: ٹوکن کی حدوں اور سیاق و سباق کی ونڈوز کو سمجھنا  
2. **سیاق و سباق سے آگاہ ایجنٹس**: ایسے ایجنٹس بنانا جو اپنی گفتگو کے سیاق و سباق کو خود سنبھالتے ہیں  
3. **سیاق و سباق کو مختصر کرنے کا پیٹرن**: گفتگو کی تاریخ کو مختصر کرنے کے لیے ٹولز کا استعمال  
4. **ایجنٹ اسکریچ پیڈ**: مستقل یادداشت جو سیاق و سباق کی کمی کے بعد بھی برقرار رہتی ہے  

## ضروریات:
- Azure OpenAI سیٹ اپ جس میں ماحول کی متغیرات ترتیب دی گئی ہوں  
- پچھلے اسباق سے بنیادی ایجنٹ تصورات کی سمجھ


## سیٹ اپ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## کیوں سیاق و سباق کا انتظام اہم ہے

ہر LLM کا ایک محدود **سیاق و سباق کی ونڈو** ہوتی ہے — زیادہ سے زیادہ ٹوکنز کی تعداد جو وہ ایک ہی درخواست میں پروسیس کر سکتا ہے۔ جیسا کہ ایک کثیر دورانیہ گفتگو بڑھتی ہے:

- **ٹوکن کی تعداد خطی طور پر بڑھتی ہے** ہر صارف کے پیغام اور معاون کے جواب کے ساتھ۔
- **پرومپٹ ٹوکنز لاگت پر غالب ہوتے ہیں** کیونکہ پوری تاریخ ہر دور میں دوبارہ بھیجی جاتی ہے۔
- بالآخر گفتگو **سیاق و سباق کی ونڈو سے تجاوز کر جاتی ہے** اور ماڈل یا تو کٹوتی کرتا ہے یا غلطی دیتا ہے۔

### سیاق و سباق کے انتظام کی حکمت عملی

| حکمت عملی | یہ کیسے کام کرتی ہے | معاہدہ |
|---|---|---|
| **کٹوتی** | پرانے پیغامات ہٹا دیں | ابتدائی سیاق ضائع ہو جاتا ہے |
| **خلاصہ سازی** | پرانے پیغامات کو خلاصے میں سمیٹنا | کچھ تفصیل ضائع ہو جاتی ہے، لیکن اہم نکات برقرار رہتے ہیں |
| **اسکریچ پیڈ / بیرونی یادداشت** | گفتگو کے باہر کلیدی حقائق محفوظ کریں | ٹول کالز کی ضرورت ہوتی ہے، لیکن کسی بھی کمی کو برداشت کر سکتا ہے |

اس نوٹ بک میں ہم **خلاصہ سازی** کو **اسکریچ پیڈ ٹول** کے ساتھ ملاتے ہیں تاکہ ایجنٹ گفتگو کی تاریخ کو مختصر کرنے کے باوجود تسلسل برقرار رکھ سکے۔


## سیاق و سباق سے واقف ایجنٹ تیار کرنا


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## لمبی گفتگو کا مشاہدہ کرنا

آئیے ایک کثیر مرحلہ گفتگو کے ذریعے چلتے ہیں تاکہ دیکھیں کہ سیاق و سباق کس طرح جمع ہوتا ہے۔ ایجنٹ کو چاہیے کہ وہ اہم تفصیلات (ترجیحات، بجٹ، سفر کی تاریخیں) کو مرحلوں کے دوران یاد رکھے اور تسلسل کا مظاہرہ کرے۔


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

نوٹس کریں کہ ایجنٹ کیسے پہلے کے مکالموں سے سیاق و سباق برقرار رکھتا ہے — اسے جاپان، سوشی، مندروں، فوٹو گرافی، $3000 بجٹ، تنہا سفر، اور اپریل کے وقت کا علم ہے۔ ایک مختصر گفتگو میں یہ اچھا کام کرتا ہے، لیکن جیسے جیسے گفتگو بڑھتی ہے پوری تاریخ دوبارہ بھیجنا مہنگا پڑ جاتا ہے۔

آئیے مزید مکالمے کے ذریعے سیاق و سباق کے جمع ہونے کو دیکھتے ہیں:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## سیاق و سباق کا خلاصہ بنانے کا نمونہ

جیسے جیسے بات چیت بڑھتی جاتی ہے، ہم **خلاصہ سازی کے آلے** کو استعمال کر سکتے ہیں تاکہ جمع شدہ سیاق و سباق کو ایک مختصر انداز میں ترتیب دیا جا سکے۔ ایجنٹ اس آلے کو اہم ترجیحات کو ریکارڈ کرنے کے لئے بلاتا ہے تاکہ اگر پرانے پیغامات ختم بھی ہو جائیں، تو بنیادی معلومات محفوظ رہیں۔

یہ نمونہ زیادہ پیچیدہ تاریخ کو کم کرنے کے لیے بنیادی بلاک ہے:
1. ایجنٹ بات چیت سے اہم حقائق کی نشاندہی کرتا ہے
2. وہ خلاصہ سازی کے آلے کو انہیں محفوظ کرنے کے لیے بلاتا ہے
3. پرانے پیغامات کو محفوظ طریقے سے حذف کیا جا سکتا ہے کیونکہ خلاصہ وہ چیزیں قید کر لیتا ہے جو اہم ہیں

ذیل میں ہم `summarize_preferences` ٹول کی تعریف کرتے ہیں جسے ایجنٹ بلاتا ہے تاکہ جو کچھ اس نے سیکھا ہے اس کا مختصر خلاصہ ریکارڈ کیا جا سکے۔


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## خلاصہ

اس سبق میں آپ نے Microsoft Agent Framework استعمال کرتے ہوئے طویل دورانیے والی ایجنٹ کی بات چیت میں کانٹیکسٹ کو مینج کرنا سیکھا:

### کلیدی تصورات
- **کانٹیکسٹ ونڈوز محدود ہوتی ہیں** — بات چیت کی تاریخ کا ہر ٹوکن قیمت رکھتا ہے اور حد میں شمار ہوتا ہے۔
- **خلاصہ سازی کے آلات** ایجنٹ کو جمع شدہ کانٹیکسٹ کو مختصر خلاصوں میں تبدیل کرنے دیتے ہیں، تاکہ ٹوکن کی مقدار کم ہو اور اہم معلومات برقرار رہے۔
- **ایجنٹ سکریچ پیڈز** مستقل بیرونی یادداشت فراہم کرتے ہیں جو کسی بھی بات چیت کی کٹائی کے بعد بھی برقرار رہتی ہے۔

### آپ نے کیا بنایا
- ایک **کانٹیکسٹ آگاہ ایجنٹ** جو کثیر دورانیے کی بات چیت میں تسلسل برقرار رکھتا ہے
- ایک **خلاصہ سازی کا آلہ** (`summarize_preferences`) جو اہم صارف کی تفصیلات کو مختصر انداز میں ریکارڈ کرتا ہے
- ایک **کثیر دورانیے کی بات چیت** جو کانٹیکسٹ کی بحالی اور تبدیلی کا مظاہرہ کرتا ہے

### حقیقی دنیا میں استعمال
- **کسٹمر سروس بوٹس**: طویل سپورٹ سیشنز میں ترجیحات یاد رکھتے ہیں
- **ذاتی معاونین**: جاری منصوبوں کو بغیر دوبارہ کانٹیکسٹ بیان کیے ٹریک کرتے ہیں
- **تعلیمی اساتذہ**: متعدد تعاملات میں طلباء کی پیش رفت کو برقرار رکھتے ہیں

### اگلے اقدامات
- مکمل سکریچ پیڈ آلہ فائل پر مبنی مستقل مزاجی کے ساتھ نافذ کریں
- خلاصہ سازی کے بعد خودکار تاریخ کو مختصر کرنا شامل کریں
- سمینٹک میموری تلاش کے لیے ویکٹر ڈیٹا بیسز کے ساتھ ملاپ کریں
- ایسے ایجنٹس بنائیں جو کئی دنوں بعد مکمل کانٹیکسٹ کے ساتھ بات چیت دوبارہ شروع کر سکیں


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ذمہ داری سے استثنا**:
اس دستاویز کا ترجمہ AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے کیا گیا ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم یاد رکھیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنی مادری زبان میں آفیشل ماخذ سمجھی جانی چاہیے۔ اہم معلومات کے لیے پیشہ ورانہ انسانی ترجمہ تجویز کیا جاتا ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تعبیر کی ذمہ داری ہم پر عائد نہیں ہوگی۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
